# AIC2026 — Kaggle Notebook: Generate All Vector Embeddings (GPU)

**Mục đích:** Sinh đồng thời image/ocr/asr embeddings trên GPU Kaggle.

**Đầu ra:** File `unified_dataset.zip` tải về máy local nạp vào Qdrant DB.

In [ ]:
# 1. Install dependencies
!pip install -q transformers pillow soundfile accelerate torch numpy pandas pyarrow pyyaml

import os
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clone Private Repo & Import Codebase Modules (Gộp chung trong 1 cell)
import os
import sys
import importlib
from pathlib import Path

GITHUB_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")
    print("🔑 Lấy thành công GITHUB_TOKEN từ Kaggle Secrets!")
except Exception as e:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")

repo_dir = Path("/kaggle/working/AIC2026").absolute()
if not (repo_dir / "shared").exists() and not Path("shared").exists():
    if GITHUB_TOKEN:
        repo_url = f"https://{GITHUB_TOKEN}@github.com/ThanhDanh1510/Super-Image-Retrieval-System-for-AIC2026.git"
    else:
        repo_url = "https://github.com/ThanhDanh1510/Super-Image-Retrieval-System-for-AIC2026.git"
    print("Executing git clone...")
    !git clone {repo_url} {repo_dir}

if repo_dir.exists():
    os.chdir(str(repo_dir))
    if str(repo_dir) not in sys.path:
        sys.path.insert(0, str(repo_dir))

if Path(".").exists() and str(Path(".").absolute()) not in sys.path:
    sys.path.insert(0, str(Path(".").absolute()))

print(f"Working Directory: {os.getcwd()}")
print(f"sys.path[0]: {sys.path[0]}")

from shared.models.image_embedding_wrapper import ImageEmbeddingWrapper
from shared.models.text_embedding_wrapper import TextEmbeddingWrapper

embed_mod = importlib.import_module("ingestion.skeletons.01c_embedding_generation")
run_embedding_generation = embed_mod.run_embedding_generation
print("✅ IMPORT THÀNH CÔNG TẤT CẢ CODE MODULES!")

In [ ]:
# 3. Thiết lập đường dẫn Input & Output
import zipfile
import numpy as np
import pandas as pd

OUTPUT_DIR = Path("/kaggle/working/unified_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

input_dir = None
for p in Path("/kaggle/input").glob("*"):
    if (p / "scene_metadata.parquet").exists():
        input_dir = p
        break

if not input_dir:
    for p in [Path("/kaggle/input/aic2026-unified-raw"), Path("/kaggle/working/unified_raw")]:
        if p.exists():
            input_dir = p
            break

print(f"INPUT DIR: {input_dir}")
assert input_dir and input_dir.exists(), "Không tìm thấy thư mục input chứa scene_metadata.parquet!"

In [ ]:
# 4. Chạy Pipeline sinh Embeddings bằng GPU
metadata_path = str(input_dir / "scene_metadata.parquet")
ocr_path = str(input_dir / "ocr_text.parquet")
asr_path = str(input_dir / "asr_text.parquet")
keyframe_dir = str(input_dir / "keyframes")

run_embedding_generation(
    metadata_path=metadata_path,
    ocr_path=ocr_path,
    asr_path=asr_path,
    keyframe_dir=keyframe_dir,
    output_dir=str(OUTPUT_DIR),
    device="cuda" if torch.cuda.is_available() else "cpu",
)

In [ ]:
# 5. Copy parquets và đóng gói thành unified_dataset.zip
import shutil

shutil.copy2(metadata_path, OUTPUT_DIR / "scene_metadata.parquet")
shutil.copy2(ocr_path, OUTPUT_DIR / "ocr_text.parquet")
shutil.copy2(asr_path, OUTPUT_DIR / "asr_text.parquet")

zip_filename = "/kaggle/working/unified_dataset.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file in OUTPUT_DIR.rglob("*"):
        if file.is_file():
            zipf.write(file, arcname=file.relative_to(OUTPUT_DIR))

print(f"✅ HOÀN THÀNH VÀ ĐÓNG GÓI: {zip_filename}")
print(f"Size: {os.path.getsize(zip_filename) / (1024*1024):.2f} MB")